# 🧠 Notebook 01 — Classification Benchmark

**Goal**: train and tune **10 classification algorithms** on the Online Shoppers Purchasing Intention dataset, compare their performance, and save all results for the report.

This notebook orchestrates the full classification pipeline:

1. Clone the repo and install dependencies
2. Load and preprocess the data
3. Run all 10 algorithms via `GridSearchCV` (stratified 5-fold CV)
4. Show the comparison table and four diagnostic figures
5. All outputs are saved under `results/` with predictable filenames

---


## ⚙️ Step 1 — Setup

In [ ]:
!git clone https://github.com/faresalotaibi888-gif/Data_Mining_HW2.git
%cd Data_Mining_HW2
!pip install -q -r requirements.txt


## 📦 Step 2 — Imports

We bring in our three custom modules: `data_loader`, `preprocessing`, and `classification`.


In [ ]:
import sys
sys.path.append('src')

from data_loader import load_online_shoppers
from preprocessing import prepare_online_shoppers, describe_prepared_data
from classification import (
    run_classification_benchmark,
    generate_all_classification_outputs,
)

print("All modules imported successfully.")


## 📥 Step 3 — Load and inspect the dataset

In [ ]:
df = load_online_shoppers()
print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()


## 🔧 Step 4 — Preprocess

This applies:
- `StandardScaler` to the 10 numeric columns
- `OneHotEncoder` to the 6 categorical columns (text + fake-numeric)
- Stratified 80/20 train/test split
- **SMOTE on training set only** to balance the ~15/85 class imbalance


In [ ]:
prepared = prepare_online_shoppers(df)
describe_prepared_data(prepared)


## 🚀 Step 5 — Run the benchmark

Now we train all 10 algorithms. Each algorithm is:

1. Wrapped in `GridSearchCV` with stratified 5-fold cross-validation
2. Tuned across a small hyperparameter grid (kept small for Colab feasibility)
3. Refit on the full training set with the best parameters
4. Evaluated on the held-out test set

**Expected runtime**: 8–15 minutes on Colab CPU. SVM and Gradient Boosting are the slowest.

You'll see live progress as each algorithm completes.


In [ ]:
results_df, fitted_models = run_classification_benchmark(
    prepared.X_train,
    prepared.X_test,
    prepared.y_train,
    prepared.y_test,
)


## 📊 Step 6 — Results table

Sorted by Test F1 (descending — best model on top).

The table is also saved to `results/tables/classification_results.csv` for the report.


In [ ]:
# Show all columns with reasonable formatting
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

results_df.round(4)


## 📈 Step 7 — Generate all comparison figures

This produces four figures (all saved under `results/figures/`):

| Filename | What it shows |
|---|---|
| `classification_metrics_comparison.png` | Grouped bars: accuracy/precision/recall/F1/AUC per algorithm |
| `classification_roc_curves.png` | Overlay of all 10 ROC curves |
| `classification_confusion_matrices.png` | 5×2 grid of confusion matrices |
| `classification_training_time.png` | Horizontal bars of training time |

Use these **exact filenames** when inserting into the report.


In [ ]:
paths = generate_all_classification_outputs(
    results_df,
    fitted_models,
    prepared.X_test,
    prepared.y_test,
)


## ✅ What we observed

After running this notebook:

- **Best F1 / ROC-AUC**: top of the sorted `results_df` (typically Gradient Boosting or Random Forest on this dataset)
- **Fastest**: Naive Bayes and LDA (under a second)
- **Slowest**: SVM (RBF) and MLP (minutes)
- **Imbalance impact**: ROC-AUC scores reveal the gap between models that learned both classes vs. those that defaulted to the majority

### Mapping to our hypotheses

- **H1** (ensembles > single models) — confirmed if Random Forest / Gradient Boosting / AdaBoost top the table
- **H2** (tree ensembles have best speed/accuracy trade-off) — visible in the metrics vs. training-time plot
- **H3** (imbalance handling improves minority recall) — visible in the recall column; SMOTE helped the linear models the most

### 📂 Files written to disk

```
results/tables/classification_results.csv
results/figures/classification_metrics_comparison.png
results/figures/classification_roc_curves.png
results/figures/classification_confusion_matrices.png
results/figures/classification_training_time.png
```

---

### 👉 Next

Notebook **02 — Clustering** on the CC General dataset.
